In [5]:

import pandas as pd

# Read CSV
df = pd.read_csv(
    r'C:\Users\user\Desktop\sql\Ikea\script\4.RetailCleaned results.csv',
    header=None,
    dtype={0: 'string'},
    low_memory=False
)

# Assign column names
df.columns = [
    'Invoice',
    'StockCode',
    'Description',
    'Quantity',
    'InvoiceDate',
    'Price',
    'Customer_ID',
    'Country',
    'SourceSheet'
]

# Convert InvoiceDate into datetime
df['InvoiceDate'] = pd.to_datetime(
    df['InvoiceDate'],
    errors='coerce',
    dayfirst=True
)

# Convert numerical columns
df['Quantity'] = pd.to_numeric(
    df['Quantity'], errors='coerce'
)

df['Price'] = pd.to_numeric(
    df['Price'], errors='coerce'
)

df['Customer_ID'] = pd.to_numeric(
    df['Customer_ID'], errors='coerce'
)

# Check dataset
print(df.shape)

df.info()

print(df.head())


(1033036, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1033036 non-null  string        
 1   StockCode    1033036 non-null  object        
 2   Description  1028761 non-null  object        
 3   Quantity     1033036 non-null  int64         
 4   InvoiceDate  423896 non-null   datetime64[ns]
 5   Price        1033036 non-null  float64       
 6   Customer_ID  797885 non-null   float64       
 7   Country      1033036 non-null  object        
 8   SourceSheet  1033036 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4), string(1)
memory usage: 70.9+ MB
  Invoice StockCode                         Description  Quantity  \
0  489434     21232      STRAWBERRY CERAMIC TRINKET BOX        24   
1  489434     21523  FANCY FONT HOME SWEET HOME DOORMAT        10   
2  489434     

In [6]:
### FILTER AND CALCULATE RECENCY,FREQUENCYAND MONETARY

In [10]:
rfm_data=df[df['Customer_ID'].notna()& df['Quantity']>0].copy()
snapshot_date=pd.to_datetime('2011-12-10')


rfm=rfm_data.groupby('Customer_ID').agg(
           Recency=('InvoiceDate',lambda x:(snapshot_date-x.max()).days),
           Frequency=('Invoice','nunique'),
           Monetary=('Quantity',lambda x: (x*rfm_data.loc[x.index,'Price']).sum())
).reset_index()

print(rfm)




      Customer_ID  Recency  Frequency  Monetary
0         12346.0    608.0         16    -43.86
1         12347.0    158.0          5    188.85
2         12348.0    219.0          4    121.00
3         12349.0    971.0          5   1122.10
4         12350.0    310.0          1     40.00
...           ...      ...        ...       ...
5221      18283.0     59.0         21   1579.50
5222      18284.0    547.0          2     35.50
5223      18285.0      NaN          1     50.00
5224      18286.0      NaN          2    207.75
5225      18287.0      NaN          2    130.55

[5226 rows x 4 columns]


In [16]:

# STEP 1: Check missing values
print(rfm[['Recency', 'Frequency', 'Monetary']].isna().sum())

# STEP 2: Remove customers with incomplete RFM metrics
rfm = rfm.dropna(
    subset=['Recency', 'Frequency', 'Monetary']
).copy()

# STEP 3: Calculate RFM scores

rfm['R_Score'] = pd.qcut(
    rfm['Recency'].rank(method='first'),
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

rfm['F_Score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm['M_Score'] = pd.qcut(
    rfm['Monetary'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)


# STEP 4: Create customer segmentation function

def segment_customer(row):

    if (row['R_Score'] >= 4 and
        row['F_Score'] >= 4 and
        row['M_Score'] >= 4):

        return 'Champion'

    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal'

    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk'

    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
        return 'New'

    elif row['R_Score'] == 3 and row['F_Score'] <= 2:
        return 'Needs Attention'

    else:
        return 'Lost'


# STEP 5: Assign customer segments

rfm['Segment'] = rfm.apply(
    segment_customer,
    axis=1
)


# STEP 6: Calculate inactivity-based churn indicator

rfm['IsChurned'] = (
    rfm['Recency'] > 90
).astype(int)


# STEP 7: Display final dataset

print(rfm.head())

# Check customer segments
print(rfm['Segment'].value_counts())

# Check churn distribution
print(rfm['IsChurned'].value_counts())


Recency      1525
Frequency       0
Monetary        0
dtype: int64
   Customer_ID  Recency  Frequency  Monetary  R_Score  F_Score  M_Score  \
0      12346.0    608.0         16    -43.86        1        5        1   
1      12347.0    158.0          5    188.85        4        3        3   
2      12348.0    219.0          4    121.00        3        3        3   
3      12349.0    971.0          5   1122.10        1        3        5   
4      12350.0    310.0          1     40.00        3        1        1   

           Segment  IsChurned  
0          At Risk          1  
1            Loyal          1  
2            Loyal          1  
3          At Risk          1  
4  Needs Attention          1  
Segment
Loyal              859
Lost               809
Champion           690
At Risk            671
New                356
Needs Attention    316
Name: count, dtype: int64
IsChurned
1    2729
0     972
Name: count, dtype: int64


Recency      1525
Frequency       0
Monetary        0
dtype: int64
Recency       245
Frequency      69
Monetary     4306
dtype: int64
           Recency    Frequency      Monetary
count  3701.000000  5226.000000   5226.000000
mean    284.933802     5.621891    282.519994
std     237.631486    12.171744   1228.156606
min      -1.000000     1.000000 -38320.500000
25%      89.000000     1.000000     44.462500
50%     218.000000     3.000000    117.620000
75%     454.000000     6.000000    316.267500
max    1061.000000   421.000000  33668.930000


In [25]:
##Merge rfm results back into transaction and export

# Merge customer-level RFM data into transaction-level data

final_df = df.merge(
    rfm,
    on='Customer_ID',
    how='left'
)

final_df.to_excel('EcommerceDasshboardData.xlsx',index=False)
print(final_df)
print(final_df.shape)



         Invoice StockCode                         Description  Quantity  \
0         489434     21232      STRAWBERRY CERAMIC TRINKET BOX        24   
1         489434     21523  FANCY FONT HOME SWEET HOME DOORMAT        10   
2         489434     21871                 SAVE THE PLANET MUG        24   
3         489434     22041        RECORD FRAME 7" SINGLE SIZE         48   
4         489434     22064          PINK DOUGHNUT TRINKET POT         24   
...          ...       ...                                 ...       ...   
1033031  C581490     23144     ZINC T-LIGHT HOLDER STARS SMALL       -11   
1033032  C581499         M                              Manual        -1   
1033033  C581568     21258          VICTORIAN SEWING BOX LARGE        -5   
1033034  C581569     20979       36 PENCILS TUBE RED RETROSPOT        -5   
1033035  C581569     84978    HANGING HEART JAR T-LIGHT HOLDER        -1   

                InvoiceDate   Price  Customer_ID         Country SourceSheet  \
0      